# Fig. A11 — Two-particle Floquet spectrum vs $V$

Two-particle quasienergies of the closed chain with $N = 2$ cells (7 sites, 21 two-particle states) for two same-spin fermions, as a
function of the nearest-neighbor interaction $V$; antisymmetric driving, $\omega = 8$, $\tilde A = 1.2$, $J = 1$, $\phi = \pi$.
The quasienergies are obtained by exact diagonalization of the one-period Floquet operator, and every level is followed continuously
in $V$ with `sort_bands` (overlap of the Floquet modes at neighboring $V$). For every level we store

- its weight on the end sites, $\langle \hat N_{C_1} + \hat N_{C_3}\rangle_\beta / 2$;
- the spectral weight of the initial state $\ket{\Psi_0} = \hat C_1^\dagger \hat C_3^\dagger\ket{0}$, $|\langle \Phi_\beta | \Psi_0\rangle|^2$;
- its flat-band character $\langle \hat c_{\phi_0}^\dagger \hat c_{\phi_0}\rangle_\beta$, the occupation of the single-particle zero (flat)
  Floquet mode $|\phi_0\rangle$. For a Slater determinant of single-particle Floquet modes it is 1 if one of the particles is in the flat
  band and 0 otherwise.

Output: `Data/FigA11_two_particle_spectrum_vs_V.npz`

In [1]:
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parents[1]                 # .../paper_reproduction
DATA = ROOT / 'Data'

from diamond_chain import finite_chain, onsite_drive, square_wave_floquet_operator, floquet_modes
from two_particles import generate_psi0, mode_occupation, two_particle_quasienergies_vs_V
from data_io import save_data

In [2]:
J = 1.0
phi = np.pi
zeta = -1
Atilde = 1.2                                  # A = Atilde * w
w = 8
V = 0.2                                       # value of Fig. 7, marked in the figure

Ncells = 2                                    # 7 sites: C1 U1 D1 C2 U2 D2 C3
Nup, Ndown = 2, 0                             # two same-spin fermions
psi0_binary = '1000001|0000000'               # one fermion on C1 and one on C3
psi0 = generate_psi0(psi0_binary, Ncells, Nup, Ndown).full().ravel()

Vvals = np.linspace(0, 0.5, 201)

Single-particle zero (flat) Floquet mode of the same chain, used for the flat-band character.

In [3]:
e_single, modes_single = floquet_modes(square_wave_floquet_operator(finite_chain(Ncells, phi, J), onsite_drive(Ncells, zeta),
                                                                     Atilde * w, w), w)
flat_mode = modes_single[:, np.argmin(np.abs(e_single))]

In [4]:
e_vs_V, end_weight_vs_V, initial_weight_vs_V, modes_vs_V = two_particle_quasienergies_vs_V(Vvals, Ncells, Nup, Ndown,
                                                                                          Atilde * w, w, psi0, zeta, J, phi)
flat_weight_vs_V = np.array([mode_occupation(modes_V, flat_mode) for modes_V in modes_vs_V])

In [5]:
save_data(DATA / 'FigA11_two_particle_spectrum_vs_V.npz',
          parameters={'Ncells': Ncells, 'Nup': Nup, 'Ndown': Ndown, 'J': J, 'phi': phi, 'w': w, 'A': Atilde * w,
                      'zeta': zeta, 'psi0_binary': psi0_binary, 'V_highlight': V},
          Vvals=Vvals, quasienergies=e_vs_V, end_weight=end_weight_vs_V, initial_weight=initial_weight_vs_V,
          flat_weight=flat_weight_vs_V)

Saved Data/FigA11_two_particle_spectrum_vs_V.npz: Vvals, quasienergies, end_weight, initial_weight, flat_weight
